In [ ]:
import json
import os
from pathlib import Path

EMBEDDED_ABLATION_CONFIGS = []
WANDB_API_KEY = "__WANDB_API_KEY_PLACEHOLDER__"

DEFAULT_CONFIG = {
    "repo_url": "https://github.com/sontungkieu/shortcut-models",
    "branch": "gmm",
    "dataset_ref": "codemaivanngu/shortcut-celebahq256",
    "dataset_name": "celebahq256",
    "tfds_data_dir": "/root/tensorflow_datasets",
    "dataset_download_dir": "/kaggle/working/shortcut_dataset",
    "batch_size": 64,
    "gmm_fit_samples": 32768,
    "gmm_valid_samples": 4096,
    "gmm_num_modes": 64,
    "gmm_em_iters": 25,
    "gmm_em_restarts": 1,
    "gmm_em_chunk_size": 128,
    "gmm_min_std": 0.0,
    "gmm_min_std_data_frac": 1.0,
    "gmm_pi_prior_type": "kl",
    "gmm_pi_prior_strength": 512.0,
    "gmm_pi_kl_steps": 100,
    "gmm_pi_kl_lr": 0.2,
    "gmm_var_prior_type": "none",
    "gmm_var_prior_strength": 0.0,
    "gmm_var_prior_target_var": 1.0,
    "gmm_standardize_data": 0,
    "wandb_project": "shortcut",
    "run_name": "gmm_ablation_manual",
    "jax_runtime": "cuda12",
}
CONFIGS = [dict(DEFAULT_CONFIG, **config) for config in EMBEDDED_ABLATION_CONFIGS] or [DEFAULT_CONFIG]

if WANDB_API_KEY and not WANDB_API_KEY.startswith("__"):
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB2")
    except Exception:
        pass

os.environ["MPLBACKEND"] = "agg"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["JAX_TRACEBACK_FILTERING"] = "off"
print(f"Batch configs: {len(CONFIGS)}")
print(json.dumps([config["run_name"] for config in CONFIGS], indent=2))


In [ ]:
!pip install -q kaggle "protobuf<4" tfds apache_beam mlcroissant
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] += ":/root/.local/bin"


In [ ]:
from pathlib import Path
import shutil

CONFIG0 = CONFIGS[0]
DATASET_REF = CONFIG0["dataset_ref"]
DOWNLOAD_DIR = Path(CONFIG0["dataset_download_dir"])
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d {DATASET_REF} -p {str(DOWNLOAD_DIR)} --unzip

if (DOWNLOAD_DIR / "tensorflow_datasets").exists():
    target = Path("/root/tensorflow_datasets")
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(DOWNLOAD_DIR / "tensorflow_datasets", target)

%cd /kaggle/working
if not Path("tfds_builders").exists():
    !git clone https://github.com/kvfrans/tfds_builders.git
%cd tfds_builders/celebahq256
!env PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python tfds build


In [ ]:
from pathlib import Path
import shutil

CONFIG0 = CONFIGS[0]
%cd /kaggle/working
if not Path("shortcut-models").exists():
    !git clone {CONFIG0["repo_url"]} shortcut-models
%cd shortcut-models
!git fetch --all
!git checkout {CONFIG0["branch"]}
!git pull
!uv sync 1>sync_out.txt 2>sync_err.txt

if CONFIG0.get("jax_runtime") == "cuda12":
    !uv pip install "jax[cuda12]==0.5.3" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html 1>jax_cuda_out.txt 2>jax_cuda_err.txt

source_data = Path(CONFIG0["dataset_download_dir"]) / "data"
if source_data.exists():
    target_data = Path("/kaggle/working/shortcut-models/data")
    if target_data.exists():
        shutil.rmtree(target_data)
    shutil.copytree(source_data, target_data)


In [ ]:
import json
import subprocess
import traceback
from pathlib import Path

batch_root = Path("/kaggle/working/gmm_ablation_batch")
batch_root.mkdir(parents=True, exist_ok=True)
batch_summary_path = batch_root / "batch_summary.jsonl"

def append_summary(row):
    with open(batch_summary_path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True) + "\n")

for index, CONFIG in enumerate(CONFIGS):
    RUN_NAME = CONFIG["run_name"]
    base_dir = batch_root / RUN_NAME
    diag_dir = base_dir / "diagnostics"
    diag_dir.mkdir(parents=True, exist_ok=True)
    gmm_stats_path = base_dir / "gmm_stats.npz"
    prep_cmd = [
        "uv", "run", "data_prep.py",
        "--dataset_name", CONFIG["dataset_name"],
        "--tfds_data_dir", CONFIG["tfds_data_dir"],
        "--batch_size", str(CONFIG["batch_size"]),
        "--gmm_save_path", str(gmm_stats_path),
        "--gmm_latent_cache_path", str(base_dir / "gmm_latents.dat"),
        "--gmm_num_modes", str(CONFIG["gmm_num_modes"]),
        "--gmm_fit_samples", str(CONFIG["gmm_fit_samples"]),
        "--gmm_valid_samples", str(CONFIG["gmm_valid_samples"]),
        "--gmm_em_iters", str(CONFIG["gmm_em_iters"]),
        "--gmm_em_restarts", str(CONFIG["gmm_em_restarts"]),
        "--gmm_init_seed", "0",
        "--gmm_standardize_data", str(CONFIG["gmm_standardize_data"]),
        "--gmm_standardize_eps", "1e-6",
        "--gmm_pi_prior_type", CONFIG["gmm_pi_prior_type"],
        "--gmm_pi_prior_strength", str(CONFIG["gmm_pi_prior_strength"]),
        "--gmm_pi_kl_steps", str(CONFIG["gmm_pi_kl_steps"]),
        "--gmm_pi_kl_lr", str(CONFIG["gmm_pi_kl_lr"]),
        "--gmm_var_prior_type", CONFIG["gmm_var_prior_type"],
        "--gmm_var_prior_strength", str(CONFIG["gmm_var_prior_strength"]),
        "--gmm_var_prior_target_var", str(CONFIG["gmm_var_prior_target_var"]),
        "--gmm_min_std", str(CONFIG["gmm_min_std"]),
        "--gmm_min_std_data_frac", str(CONFIG["gmm_min_std_data_frac"]),
        "--gmm_kmeanspp_init", "1",
        "--gmm_em_chunk_size", str(CONFIG["gmm_em_chunk_size"]),
        "--gmm_keep_latent_cache", "0",
        "--metrics_output_path", str(diag_dir / "gmm_metrics.json"),
        "--gmm_em_metrics_output_path", str(diag_dir / "gmm_em_metrics.jsonl"),
        "--wandb.name", f"prep_{RUN_NAME}",
    ]
    row = {
        "index": index,
        "run_name": RUN_NAME,
        "base_dir": str(base_dir),
        "metrics_path": str(diag_dir / "gmm_metrics.json"),
        "config": CONFIG,
    }
    try:
        print(f"[{index + 1}/{len(CONFIGS)}] {RUN_NAME}")
        with open(diag_dir / "gmm_prep_stdout.txt", "w") as out, open(diag_dir / "gmm_prep_stderr.txt", "w") as err:
            result = subprocess.run(prep_cmd, stdout=out, stderr=err, check=False)
        row["returncode"] = result.returncode
        metrics_path = diag_dir / "gmm_metrics.json"
        if metrics_path.exists():
            metrics = json.loads(metrics_path.read_text())
            for key in [
                "train_nll", "valid_nll", "pi_entropy_normalized",
                "dead_component_count", "count_gap", "count_ratio",
                "component_var_mean", "component_var_min", "var_floor_hit_rate",
                "data_variance_mean",
            ]:
                if key in metrics:
                    row[key] = metrics[key]
        append_summary(row)
    except Exception as exc:
        row["returncode"] = -1
        row["error"] = repr(exc)
        row["traceback"] = traceback.format_exc()[-4000:]
        append_summary(row)
        print(row["traceback"])

print("Batch summary:", batch_summary_path)


In [ ]:
from pathlib import Path
batch_root = Path("/kaggle/working/gmm_ablation_batch")
print("Batch output:", batch_root)
!find {str(batch_root)} -maxdepth 4 -type f | sort
